# 05 — Analysis

Evaluates all three model stages — base `distilgpt2`, SFT, and PPO —
by generating bios for the same prompts and scoring with the reward model.
Produces a comparison chart, example side-by-side outputs, and a summary
HTML report.

Requires all checkpoints from previous notebooks.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jonasneves/aipi590-challenge-2/blob/main/notebooks/05_analysis.ipynb)

In [ ]:
# Setup — install dependencies, clone repo, configure environment
!pip install -q trl peft transformers datasets accelerate bitsandbytes

import os
from google.colab import userdata
os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")

import sys
sys.path.insert(0, "/content/aipi590-challenge-2")

from src.colab_utils import prepare_notebook, publish_artifacts

repo_root, paths = prepare_notebook("aipi590-challenge-2")
print(f"Repo root: {repo_root}")

sft_checkpoint = paths["results"] / "sft_checkpoint"
ppo_checkpoint = paths["results"] / "ppo_checkpoint"
rm_checkpoint  = paths["results"] / "reward_checkpoint"

for ckpt, name in [(sft_checkpoint, "SFT"), (ppo_checkpoint, "PPO"), (rm_checkpoint, "Reward")]:
    status = "found" if ckpt.exists() else "MISSING — falling back to base model"
    print(f"  {name}: {status}")

In [ ]:
# Load all three models and the reward model scorer
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

BASE_ID = "distilgpt2"

# Shared tokenizer (all three models use same vocab)
tokenizer = AutoTokenizer.from_pretrained(BASE_ID)
tokenizer.pad_token = tokenizer.eos_token

def load_gen_model(path_or_id: str):
    m = AutoModelForCausalLM.from_pretrained(
        path_or_id,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    ).to(device).eval()
    return m

print("Loading base model...")
base_model = load_gen_model(BASE_ID)

print("Loading SFT model...")
sft_model = load_gen_model(str(sft_checkpoint) if sft_checkpoint.exists() else BASE_ID)

print("Loading PPO model...")
ppo_model = load_gen_model(str(ppo_checkpoint) if ppo_checkpoint.exists() else BASE_ID)

print("Loading reward model...")
rm_base = str(rm_checkpoint) if rm_checkpoint.exists() else BASE_ID
reward_model = AutoModelForSequenceClassification.from_pretrained(rm_base, num_labels=1)
reward_model.config.pad_token_id = tokenizer.pad_token_id
reward_model = reward_model.to(device).eval()

print("All models loaded.")


def generate_bio(model, prompt_text: str, max_new_tokens: int = 80) -> str:
    enc = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def score_bio(prompt_text: str, bio_text: str) -> float:
    full_text = prompt_text + bio_text
    enc = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        out = reward_model(**enc)
    return out.logits.squeeze().item()

In [ ]:
# Generate 10 bios per model and score with reward model
import json
from src.dataset import format_sft_prompt

EVAL_PROMPTS = [
    "Software engineer, 28, NYC, climbs on weekends, makes sourdough",
    "Nurse, 26, Chicago, reads sci-fi, runs half marathons",
    "High school history teacher, 31, Austin, plays guitar, homebrews beer",
    "UX designer, 27, SF, surfs, obsessed with typography",
    "Grad student in ecology, 25, Seattle, birdwatcher, plays chess",
    "Pastry chef, 29, New Orleans, jazz fan, does pottery",
    "Aerospace engineer, 30, LA, amateur astronomer, plays volleyball",
    "Documentary filmmaker, 27, Portland, vegan, forages for mushrooms",
    "Financial analyst, 32, Boston, into CrossFit, reads philosophy",
    "Veterinarian, 28, Denver, hikes fourteeners, plays piano",
]

results = {"base": [], "sft": [], "ppo": []}

for i, person in enumerate(EVAL_PROMPTS):
    prompt = format_sft_prompt(person)
    for label, model in [("base", base_model), ("sft", sft_model), ("ppo", ppo_model)]:
        bio = generate_bio(model, prompt)
        score = score_bio(prompt, bio)
        results[label].append({"person": person, "bio": bio, "score": score})

    if (i + 1) % 3 == 0:
        print(f"  Processed {i + 1}/{len(EVAL_PROMPTS)} prompts")

import numpy as np
for label in ["base", "sft", "ppo"]:
    scores = [r["score"] for r in results[label]]
    print(f"{label.upper():4s}: mean={np.mean(scores):.4f}  std={np.std(scores):.4f}")

In [ ]:
# Bar chart: mean reward score comparison across model stages
import matplotlib.pyplot as plt
import numpy as np

results_dir = paths["results"]

labels = ["Base\ndistilgpt2", "SFT", "PPO"]
means  = [np.mean([r["score"] for r in results[k]]) for k in ["base", "sft", "ppo"]]
stds   = [np.std([r["score"]  for r in results[k]]) for k in ["base", "sft", "ppo"]]
colors = ["#A8A8A8", "#4A90D9", "#5CB85C"]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(labels, means, yerr=stds, capsize=6, color=colors, edgecolor="white",
              linewidth=0.8, error_kw={"elinewidth": 1.5, "ecolor": "#555"})

for bar, mean in zip(bars, means):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(stds) * 1.1 + 0.005,
        f"{mean:.3f}",
        ha="center", va="bottom", fontsize=11, fontweight="bold"
    )

ax.set_title("Mean Reward Score by Model Stage", fontsize=13, fontweight="bold", pad=12)
ax.set_ylabel("Mean Reward Score", fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3, linestyle="--")
plt.tight_layout()

chart_path = results_dir / "reward_comparison.png"
plt.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {chart_path}")

In [ ]:
# Side-by-side examples: base vs SFT vs PPO for 3 person descriptions
EXAMPLE_INDICES = [0, 4, 8]

print("=" * 80)
for idx in EXAMPLE_INDICES:
    person = EVAL_PROMPTS[idx]
    print(f"\nPERSON: {person}\n")
    for label in ["base", "sft", "ppo"]:
        r = results[label][idx]
        score = r["score"]
        bio = r["bio"]
        print(f"  [{label.upper():3s}] (score={score:.3f})")
        print(f"        {bio}")
    print("-" * 80)

In [ ]:
# Generate results/index.html with metrics table and embedded charts
import base64
import json
import numpy as np
from pathlib import Path

def img_to_base64(path: Path) -> str:
    return base64.b64encode(path.read_bytes()).decode("utf-8")

comparison_b64 = img_to_base64(results_dir / "reward_comparison.png")

# Load prior charts if they exist
pref_dist_b64 = ""
pref_dist_path = results_dir / "preference_distribution.png"
if pref_dist_path.exists():
    pref_dist_b64 = img_to_base64(pref_dist_path)

reward_dist_b64 = ""
reward_dist_path = results_dir / "reward_score_distribution.png"
if reward_dist_path.exists():
    reward_dist_b64 = img_to_base64(reward_dist_path)

ppo_curves_b64 = ""
ppo_curves_path = results_dir / "metrics" / "ppo_training_curves.png"
if ppo_curves_path.exists():
    ppo_curves_b64 = img_to_base64(ppo_curves_path)

# Build metrics table rows
table_rows = ""
for label, display in [("base", "Base distilgpt2"), ("sft", "SFT"), ("ppo", "PPO")]:
    scores = [r["score"] for r in results[label]]
    table_rows += f"""
      <tr>
        <td>{display}</td>
        <td>{np.mean(scores):.4f}</td>
        <td>{np.std(scores):.4f}</td>
        <td>{min(scores):.4f}</td>
        <td>{max(scores):.4f}</td>
      </tr>"""

# Build example output rows
example_rows = ""
for idx in EXAMPLE_INDICES:
    person = EVAL_PROMPTS[idx]
    base_r = results["base"][idx]
    sft_r  = results["sft"][idx]
    ppo_r  = results["ppo"][idx]
    example_rows += f"""
    <div class="example">
      <h3>{person}</h3>
      <table class="bio-table">
        <tr><th>Model</th><th>Bio</th><th>Score</th></tr>
        <tr><td>Base</td><td>{base_r['bio']}</td><td>{base_r['score']:.3f}</td></tr>
        <tr><td>SFT</td><td>{sft_r['bio']}</td><td>{sft_r['score']:.3f}</td></tr>
        <tr><td class="ppo-label">PPO</td><td>{ppo_r['bio']}</td><td>{ppo_r['score']:.3f}</td></tr>
      </table>
    </div>"""

def chart_section(b64: str, title: str) -> str:
    if not b64:
        return ""
    return f"""
    <div class="chart">
      <h2>{title}</h2>
      <img src="data:image/png;base64,{b64}" alt="{title}" />
    </div>"""

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>RLHF Pipeline — Hinge Bio Generation</title>
  <style>
    body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
           max-width: 960px; margin: 40px auto; padding: 0 24px;
           background: #f9f9f9; color: #1a1a1a; }}
    h1   {{ font-size: 1.8rem; font-weight: 700; margin-bottom: 4px; }}
    h2   {{ font-size: 1.3rem; font-weight: 600; margin-top: 40px; }}
    h3   {{ font-size: 1rem; font-weight: 600; color: #444; margin-bottom: 8px; }}
    .subtitle {{ color: #666; font-size: 0.95rem; margin-bottom: 36px; }}
    table {{ width: 100%; border-collapse: collapse; margin: 12px 0 24px; }}
    th, td {{ padding: 10px 14px; text-align: left; border-bottom: 1px solid #e0e0e0; }}
    th {{ background: #f0f0f0; font-weight: 600; font-size: 0.9rem; }}
    td {{ font-size: 0.9rem; }}
    .bio-table td {{ vertical-align: top; }}
    .ppo-label {{ font-weight: 700; color: #2c7a2c; }}
    .chart img {{ max-width: 100%; border-radius: 8px;
                  box-shadow: 0 2px 8px rgba(0,0,0,0.08); margin-top: 8px; }}
    .example {{ background: white; border-radius: 8px; padding: 20px;
                margin: 16px 0; box-shadow: 0 1px 4px rgba(0,0,0,0.06); }}
  </style>
</head>
<body>
  <h1>RLHF Pipeline — Hinge Bio Generation</h1>
  <p class="subtitle">
    Distilgpt2 fine-tuned on synthetic Hinge bios (SFT) and aligned via
    Proximal Policy Optimization (PPO) using human preference annotations.
  </p>

  <h2>Reward Score Summary</h2>
  <table>
    <tr><th>Model</th><th>Mean Score</th><th>Std</th><th>Min</th><th>Max</th></tr>
    {table_rows}
  </table>

  {chart_section(comparison_b64, 'Mean Reward Score by Model Stage')}
  {chart_section(pref_dist_b64, 'Preference Distribution (Human Annotations)')}
  {chart_section(reward_dist_b64, 'Reward Model Score Distribution')}
  {chart_section(ppo_curves_b64, 'PPO Training Curves')}

  <h2>Example Outputs</h2>
  {example_rows}
</body>
</html>
"""

html_path = results_dir / "index.html"
html_path.write_text(html)
print(f"Report saved to {html_path}")

In [ ]:
# Save all results and publish
import json
import numpy as np

# Save full evaluation results as JSON
eval_results = {
    model_label: {
        "mean_score": float(np.mean([r["score"] for r in recs])),
        "std_score":  float(np.std([r["score"]  for r in recs])),
        "samples": [{"person": r["person"], "bio": r["bio"], "score": r["score"]} for r in recs],
    }
    for model_label, recs in results.items()
}

eval_path = paths["results"] / "eval_results.json"
eval_path.write_text(json.dumps(eval_results, indent=2))
print(f"Eval results saved to {eval_path}")

artifacts = [
    eval_path,
    paths["results"] / "reward_comparison.png",
    paths["results"] / "index.html",
]

publish_artifacts(
    artifacts,
    "add evaluation results, comparison chart, and HTML report",
    repo_root,
)
print("Done. All results published.")